In [ ]:
from itertools import takewhile
from pathlib import Path


def natural_sort_str(name: str):
    """Natural sort key for strings without regex (e.g., '0.2' before '0.10')."""
    parts = []
    buf = ""
    buf_is_digit = None
    for ch in name:
        is_d = ch.isdigit()
        if buf_is_digit is None or is_d == buf_is_digit:
            buf += ch
            buf_is_digit = is_d
        else:
            parts.append(int(buf) if buf_is_digit else buf)
            buf = ch
            buf_is_digit = is_d
    if buf:
        parts.append(int(buf) if buf_is_digit else buf)
    return parts


def get_leading_prefix(name: str) -> str:
    """Gets leading digits like '0' or '10' from '0_echo_state.ipynb' or '0.1_plot.png'."""
    return "".join(takewhile(str.isdigit, name))


def get_plot_number(filename: str) -> float:
    """Extracts numeric prefix like 0.1 or 0.10 from '0.10_plot.png' without regex."""
    first_part = filename.split("_", 1)[0]
    cleaned = "".join(ch for ch in first_part if ch.isdigit() or ch == ".")
    try:
        return float(cleaned)
    except ValueError:
        return 0.0


def sync_experiment_note(exp_dir: str):
    exp_path = Path(exp_dir)
    note_path = exp_path / f"{exp_path.name}.md"

    text = note_path.read_text() if note_path.exists() else "\nblah blah"

    code_files = list(exp_path.glob("*.ipynb")) + list(exp_path.glob("*.py"))
    numbered_dirs = [
        d
        for d in exp_path.iterdir()
        if d.is_dir() and d.name != "plots" and d.name and d.name[0].isdigit()
    ]
    file_names = [f.name for f in (code_files + numbered_dirs)]
    sections = sorted(file_names, key=natural_sort_str)

    for sec in sections:
        header = f"\n\n## `{sec}`"
        if header not in text:
            text += f"{header}\n- blah blah \n- **Plots:**"

    plots_dir = exp_path / "plots"
    if plots_dir.exists():
        plot_names = [p.name for p in plots_dir.glob("*.png")]
        plots = sorted(plot_names, key=natural_sort_str)

        for sec in sections:
            prefix = get_leading_prefix(sec)
            if not prefix:
                continue

            matching_plots = [
                plot for plot in plots if get_leading_prefix(plot) == prefix
            ]
            if not matching_plots:
                continue

            sec_header = f"## `{sec}`"
            sec_pos = text.find(sec_header)
            if sec_pos == -1:
                continue

            for plot in matching_plots:
                if f"./plots/{plot}" in text or f"![[{plot}" in text:
                    continue

                new_plot_num = get_plot_number(plot)
                plot_block = (
                    f"    - **{plot}**\n"
                    f"    ![{plot}](./plots/{plot})\n"
                    f"    - blah blah\n"
                )

                # Scope search to this section only (up to the next '## `')
                next_sec_pos = text.find("\n## `", sec_pos + len(sec_header))
                section_end = len(text) if next_sec_pos == -1 else next_sec_pos

                # Split section slice into lines with character offsets
                section_slice = text[sec_pos:section_end]
                lines = section_slice.splitlines(keepends=True)

                inserted = False
                offset = sec_pos
                for line in lines:
                    stripped = line.lstrip()
                    # Match plot bullet lines like '- **0.1_plot.png**'
                    if stripped.startswith("- **") and "**" in stripped[4:]:
                        end_bold = stripped.find("**", 4)
                        plot_title = stripped[4:end_bold]
                        existing_num = get_plot_number(plot_title)

                        if new_plot_num < existing_num:
                            text = text[:offset] + plot_block + text[offset:]
                            inserted = True
                            break

                    offset += len(line)

                if not inserted:
                    plots_tag = "- **Plots:**"
                    plots_tag_pos = text.find(plots_tag, sec_pos)
                    if plots_tag_pos != -1:
                        insert_pos = plots_tag_pos + len(plots_tag)
                        next_section_pos = text.find("\n\n## `", insert_pos)
                        if next_section_pos == -1:
                            text += plot_block
                        else:
                            text = (
                                text[:next_section_pos]
                                + plot_block
                                + text[next_section_pos:]
                            )

    note_path.write_text(text)
    print(f"Synced & Safely Inserted: {exp_path.name}")

In [171]:
sync_experiment_note(
    "/Users/churroc/Personal/code/disorder_reservoir_research/src/experiments/26_09_22_benchmark_mem_cap"
)
print("Obsidian note updated with latest plots!")

Synced & Safely Inserted: 26_09_22_benchmark_mem_cap
Obsidian note updated with latest plots!


In [167]:
from pathlib import Path

folder_path = Path(
    "/Users/churroc/Personal/code/disorder_reservoir_research/src/experiments"
)
subfolders = [f for f in folder_path.iterdir() if f.is_dir()]


for folder in subfolders:
    sync_experiment_note(folder)

Synced & Safely Inserted: 26_11_17_contrabillity
Synced & Safely Inserted: 26_09_22_benchmark_mem_cap
Synced & Safely Inserted: 26_10_12_slideshow
Synced & Safely Inserted: 0_summer
Synced & Safely Inserted: 26_10_13_spring_potentials
Synced & Safely Inserted: 26_09_15_fly_brain
Synced & Safely Inserted: 26_08_03_pendulum_res
Synced & Safely Inserted: 26_10_12_spring_henon_Danny_26_10_06
Synced & Safely Inserted: 26_10_10_basic_results_overview_grant
Synced & Safely Inserted: 26_10_06_anderson_local
Synced & Safely Inserted: 26_09_30_bayesian_mem_cap
Synced & Safely Inserted: 26_10_07_benchmark_rank_gr
